# exp118_spatial_neighbor_prior_confidence_gate_on_exp092 train

Train-side posthoc audit of confidence gates for applying exp114 spatial neighbor prior corrections to exp092 OOF predictions.

## Contents

1. Setup and configuration
2. Input artifact preview
3. Confidence gate audit
4. Metrics and artifacts

## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Spatial prior parent:", get_nested(config, "lineage.spatial_prior_parent"))
print("Validation:", get_nested(config, "validation.strategy"))
print("Models:", get_nested(config, "gate.exp092_models"))
print("Spatial variants:", get_nested(config, "gate.spatial_variants"))
print("Artifacts:", paths.artifacts_dir)

## 2. Input artifact preview


In [ ]:
from spatial_neighbor_prior_confidence_gate_on_exp092 import (
    EXP092_PREDICTIONS,
    EXP114_OOF,
    find_artifact,
    parse_gate_specs,
)

spatial_oof = find_artifact(EXP114_OOF, get_nested(config, "data.exp114_oof_predictions_local"))
exp092_predictions = find_artifact(EXP092_PREDICTIONS, get_nested(config, "data.exp092_predictions_local"))
gates = parse_gate_specs(config)

print("Spatial OOF:", spatial_oof)
print("Spatial OOF size bytes:", spatial_oof.stat().st_size)
print("exp092 predictions:", exp092_predictions)
print("exp092 predictions size bytes:", exp092_predictions.stat().st_size)
print("Gate policies:")
for gate in gates:
    print(" -", gate)

spatial_preview = pd.read_csv(spatial_oof, nrows=5)
display(spatial_preview[["id", "well", "true_tvt", "md_since", "likpf_mean", "xy_plus_trajectory_shape_k8_prior_tvt", "xy_plus_trajectory_shape_k8_prior_std"]])
pred_preview = pd.read_csv(exp092_predictions, nrows=5)
display(pred_preview[["id", "well", "variant", "mode", "model", "target_tvt", "pred_tvt"]])

## 3. Confidence gate audit

The audit keeps exp092 OOF predictions fixed, applies small clipped spatial-prior corrections only where target-free confidence gates pass, and scores the posthoc policies against the same OOF rows.

In [ ]:
from spatial_neighbor_prior_confidence_gate_on_exp092 import run_audit, to_jsonable

summary = run_audit(config=config, paths=paths)
print(json.dumps(to_jsonable({
    "rows": summary["rows"],
    "wells": summary["wells"],
    "best_policy": summary["best_policy"],
    "baseline_policy": summary["baseline_policy"],
    "best_by_well_delta": summary["best_by_well_delta"],
    "decision": summary["decision"],
}), indent=2, sort_keys=True))

## 4. Metrics and artifacts


In [ ]:
artifact_paths = {name: Path(path) for name, path in summary["artifacts"].items()}
for name, path in artifact_paths.items():
    print(f"{name}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

gate_metrics = pd.read_csv(artifact_paths["gate_metrics"])
by_well_delta = pd.read_csv(artifact_paths["by_well_delta"])
bucket_metrics = pd.read_csv(artifact_paths["bucket_metrics"])
display(gate_metrics.head(20))
display(by_well_delta.head(20))
display(bucket_metrics.head(20))